### SHAP

In [ ]:
# read pkl file
import pandas as pd
df_final = pd.read_pickle('********.pkl')
df_final

In [ ]:
selected_features = [
    
 'max_Heart Rate',
       'max_Non Invasive Blood Pressure diastolic',
       'max_Non Invasive Blood Pressure mean',
       'max_Non Invasive Blood Pressure systolic',
       'max_O2 saturation pulseoxymetry', 'max_Respiratory Rate',
       'max_Temperature Fahrenheit', 'mean_Heart Rate',
       'mean_Non Invasive Blood Pressure diastolic',
       'mean_Non Invasive Blood Pressure mean',
       'mean_Non Invasive Blood Pressure systolic',
       'mean_O2 saturation pulseoxymetry', 'mean_Respiratory Rate',
       'mean_Temperature Fahrenheit', 'median_Heart Rate',
       'median_Non Invasive Blood Pressure diastolic',
       'median_Non Invasive Blood Pressure mean',
       'median_Non Invasive Blood Pressure systolic',
       'median_O2 saturation pulseoxymetry', 'median_Respiratory Rate',
       'median_Temperature Fahrenheit', 'min_Heart Rate',
       'min_Non Invasive Blood Pressure diastolic',
       'min_Non Invasive Blood Pressure mean',
       'min_Non Invasive Blood Pressure systolic',
       'min_O2 saturation pulseoxymetry', 'min_Respiratory Rate',
       'min_Temperature Fahrenheit', 'q25_Heart Rate',
       'q25_Non Invasive Blood Pressure diastolic',
       'q25_Non Invasive Blood Pressure mean',
       'q25_Non Invasive Blood Pressure systolic',
       'q25_O2 saturation pulseoxymetry', 'q25_Respiratory Rate',
       'q25_Temperature Fahrenheit', 'q75_Heart Rate',
       'q75_Non Invasive Blood Pressure diastolic',
       'q75_Non Invasive Blood Pressure mean',
       'q75_Non Invasive Blood Pressure systolic',
       'q75_O2 saturation pulseoxymetry', 'q75_Respiratory Rate',
       'q75_Temperature Fahrenheit', 'variance_Heart Rate',
       'variance_Non Invasive Blood Pressure diastolic',
       'variance_Non Invasive Blood Pressure mean',
       'variance_Non Invasive Blood Pressure systolic',
       'variance_O2 saturation pulseoxymetry', 'variance_Respiratory Rate',
       'variance_Temperature Fahrenheit', 
       
       'age_at_admission', 'gender', 'race_group', 
       
       'aids', 'cerebrovascular_disease',
       #'chronic_pulmonary_disease', 
       'congestive_heart_failure', 'dementia',
       'diabetes_with_cc', 'diabetes_without_cc', 'malignant_cancer',
       'metastatic_solid_tumor', 'mild_liver_disease', 'myocardial_infarction',
       'paraplegia', 'peptic_ulcer_disease', 'peripheral_vascular_disease',
       'renal_disease', 'rheumatic_disease', 'severe_liver_disease'

 ]

In [ ]:
import numpy as np
X = df_final[selected_features].values
y = df_final['label'].values
groups = df_final['subject_id'].values

print(f"Dataset shape: {X.shape}")
print(f"Category distribution: {np.bincount(y)}")
print(f"Number of unique subjects: {len(np.unique(groups))}")

In [ ]:
import numpy as np
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_curve, roc_auc_score, accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import clone
from sklearn.svm import SVC

# Data Preparation
X = df_final[selected_features].values
y = df_final['label'].values
groups = df_final['subject_id'].values

#Split out an independent test set based on subjects
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_val_idx, test_idx = next(gss.split(X, y, groups))

X_tv = X[train_val_idx]
y_tv = y[train_val_idx]
groups_tv = groups[train_val_idx]

X_test = X[test_idx]
y_test = y[test_idx]
groups_test = groups[test_idx]

print(f"Train + Validation Set: {len(np.unique(groups_tv))} subjects, {len(y_tv)} records")
print(f"Independent Test Set:   {len(np.unique(groups_test))} subjects, {len(y_test)} records")
print(f"Positive Proportion in Test Set: {y_test.mean():.1%}")

#Build a pipeline using fixed optimal parameters
rf_base = SVC(class_weight='balanced', probability=True, random_state=42, kernel='linear', C=0.1, degree=2)

rf_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', rf_base)
])

#Perform OOF (Out-of-Fold) analysis on the Train+Validation set to find the optimal threshold.
print("\n=== Calculate OOF probabilities on the Train+Validation set and find the optimal threshold. ===")

outer_gkf = GroupKFold(n_splits=5)
oof_proba = np.zeros(len(y_tv))

for train_idx, val_idx in outer_gkf.split(X_tv, y_tv, groups_tv):
    X_train = X_tv[train_idx]
    y_train = y_tv[train_idx]
    g_train = groups_tv[train_idx]
    
    model = clone(rf_pipeline)
    model.fit(X_train, y_train)
    oof_proba[val_idx] = model.predict_proba(X_tv[val_idx])[:, 1]

# Calculate the ROC curve
fpr, tpr, thresholds = roc_curve(y_tv, oof_proba)
specificity = 1 - fpr

# Set your desired minimum sensitivity
min_sensitivity = 0.70       

#Filter out all thresholds that satisfy Sensitivity >= min_sensitivity
valid_idx = np.where(tpr >= min_sensitivity)[0]

if len(valid_idx) == 0:
    print("Warning: No threshold meets the minimum sensitivity requirement; please lower min_sensitivity.")
    optimal_threshold = 0.5
else:
    #Among the thresholds that meet the criteria, select the one with the highest specificity.
    best_idx = valid_idx[np.argmax(specificity[valid_idx])]
    optimal_threshold = thresholds[best_idx]

print(f"Target Minimum Sensitivity: {min_sensitivity:.3f}")
print(f"Final optimal threshold: {optimal_threshold:.4f}")
print(f"Corresponds to Sensitivity: {tpr[best_idx]:.4f} | Specificity: {specificity[best_idx]:.4f}")

# Train the final model on the complete Train+Validation set
final_model = clone(rf_pipeline)
final_model.fit(X_tv, y_tv)

# Make predictions on an independent test set using the optimal threshold.
print("\n=== Final performance on the independent test set (using the optimal threshold) ===")

proba_test = final_model.predict_proba(X_test)[:, 1]
pred_test = (proba_test >= optimal_threshold).astype(int)   

auc = roc_auc_score(y_test, proba_test)
acc = accuracy_score(y_test, pred_test)
f1 = f1_score(y_test, pred_test)
prec = precision_score(y_test, pred_test)
rec = recall_score(y_test, pred_test)

tn, fp, fn, tp = confusion_matrix(y_test, pred_test).ravel()
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

print(f"AUC: {auc:.4f}")
print(f"Accuracy: {acc:.1%}")
print(f"F1-Score: {f1:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall (Sensitivity): {rec:.4f}")
print(f"Sensitivity: {sensitivity:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"|Sensitivity - Specificity|: {abs(sensitivity - specificity):.4f}")
print(f"Confusion Matrix → TP={tp}  FN={fn}  TN={tn}  FP={fp}")

In [ ]:
# SHAP analysis
import shap
import matplotlib.pyplot as plt

#Prepare background data (required by KernelExplainer)
background = shap.kmeans(X, 50)     

# Create KernelExplainer
explainer = shap.KernelExplainer(
    final_model.predict_proba,     
    background,
    link="logit"                   
)

#Calculate SHAP values ​​on a separate test set
shap_values = explainer.shap_values(X_test)   

# `shap_values` is a list; for a binary SVM classification, `shap_values[1]` represents the SHAP values ​​for the positive class (label=1)
shap_values_pos = shap_values[1] if isinstance(shap_values, list) else shap_values


# Plot a SHAP bar chart (ranking feature importance)
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values_pos, X_test, feature_names=selected_features, plot_type="bar", show=False, max_display=10)
plt.title("SHAP Feature Importance (SVM)", fontsize=14)
plt.tight_layout()
plt.savefig("*****.pdf", dpi=300, bbox_inches='tight')
plt.show()

# Output the top 10 most important features
feature_importance = np.abs(shap_values_pos).mean(0)
top_idx = np.argsort(feature_importance)[::-1][:10]
print("\n=== Top 10 Most Important Features (SVM) ===")
for i, idx in enumerate(top_idx):
    print(f"{i+1:2d}. {selected_features[idx]:<30}  SHAP = {feature_importance[idx]:.4f}")

In [ ]:
# read pkl file
import pandas as pd
df_final = pd.read_pickle('*******.pkl')
df_final

In [ ]:
selected_features = [
    
 'max_Heart Rate',
       'max_Non Invasive Blood Pressure diastolic',
       'max_Non Invasive Blood Pressure mean',
       'max_Non Invasive Blood Pressure systolic',
       'max_O2 saturation pulseoxymetry', 'max_Respiratory Rate',
       'max_Temperature Fahrenheit', 'mean_Heart Rate',
       'mean_Non Invasive Blood Pressure diastolic',
       'mean_Non Invasive Blood Pressure mean',
       'mean_Non Invasive Blood Pressure systolic',
       'mean_O2 saturation pulseoxymetry', 'mean_Respiratory Rate',
       'mean_Temperature Fahrenheit', 'median_Heart Rate',
       'median_Non Invasive Blood Pressure diastolic',
       'median_Non Invasive Blood Pressure mean',
       'median_Non Invasive Blood Pressure systolic',
       'median_O2 saturation pulseoxymetry', 'median_Respiratory Rate',
       'median_Temperature Fahrenheit', 'min_Heart Rate',
       'min_Non Invasive Blood Pressure diastolic',
       'min_Non Invasive Blood Pressure mean',
       'min_Non Invasive Blood Pressure systolic',
       'min_O2 saturation pulseoxymetry', 'min_Respiratory Rate',
       'min_Temperature Fahrenheit', 'q25_Heart Rate',
       'q25_Non Invasive Blood Pressure diastolic',
       'q25_Non Invasive Blood Pressure mean',
       'q25_Non Invasive Blood Pressure systolic',
       'q25_O2 saturation pulseoxymetry', 'q25_Respiratory Rate',
       'q25_Temperature Fahrenheit', 'q75_Heart Rate',
       'q75_Non Invasive Blood Pressure diastolic',
       'q75_Non Invasive Blood Pressure mean',
       'q75_Non Invasive Blood Pressure systolic',
       'q75_O2 saturation pulseoxymetry', 'q75_Respiratory Rate',
       'q75_Temperature Fahrenheit', 'variance_Heart Rate',
       'variance_Non Invasive Blood Pressure diastolic',
       'variance_Non Invasive Blood Pressure mean',
       'variance_Non Invasive Blood Pressure systolic',
       'variance_O2 saturation pulseoxymetry', 'variance_Respiratory Rate',
       'variance_Temperature Fahrenheit', 
       
       'age_at_admission', 'gender', 'race_group', 
       


 ]

In [ ]:
import numpy as np
X = df_final[selected_features].values
y = df_final['label'].values
groups = df_final['subject_id'].values

print(f"Dataset shape: {X.shape}")
print(f"Class distribution: {np.bincount(y)}")
print(f"Number of unique subjects: {len(np.unique(groups))}")

In [ ]:
import numpy as np
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_curve, roc_auc_score, accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import clone
from sklearn.svm import SVC

# Data preparation
X = df_final[selected_features].values
y = df_final['label'].values
groups = df_final['subject_id'].values

# Split into an independent test set based on subjects
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_val_idx, test_idx = next(gss.split(X, y, groups))

X_tv = X[train_val_idx]
y_tv = y[train_val_idx]
groups_tv = groups[train_val_idx]

X_test = X[test_idx]
y_test = y[test_idx]
groups_test = groups[test_idx]

print(f"Train + Validation Set: {len(np.unique(groups_tv))} subjects, {len(y_tv)} records")
print(f"Independent Test Set:   {len(np.unique(groups_test))} subjects, {len(y_test)} records")
print(f"Proportion of positive cases in the test set: {y_test.mean():.1%}")

# Build a pipeline using fixed, optimal parameters
rf_base = SVC(class_weight='balanced', probability=True, random_state=42, kernel='linear', C=0.1, degree=2)

rf_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', rf_base)
])

# Perform out-of-fold (OOF) analysis on the train+validation set to determine the optimal threshold
print("\n=== Calculate OOF probabilities on the train+validation set and identify the optimal threshold ===")

outer_gkf = GroupKFold(n_splits=5)
oof_proba = np.zeros(len(y_tv))

for train_idx, val_idx in outer_gkf.split(X_tv, y_tv, groups_tv):
    X_train = X_tv[train_idx]
    y_train = y_tv[train_idx]
    g_train = groups_tv[train_idx]
    
    model = clone(rf_pipeline)
    model.fit(X_train, y_train)
    oof_proba[val_idx] = model.predict_proba(X_tv[val_idx])[:, 1]

#Calculate the ROC curve
fpr, tpr, thresholds = roc_curve(y_tv, oof_proba)
specificity = 1 - fpr

# Set your desired minimum sensitivity
min_sensitivity = 0.65      

# Filter for all thresholds that satisfy sensitivity ≥ min_sensitivity
valid_idx = np.where(tpr >= min_sensitivity)[0]

if len(valid_idx) == 0:
    print("Warning: No threshold meets the minimum sensitivity requirement; please lower the `min_sensitivity` value.")
    optimal_threshold = 0.5
else:
    # Among the thresholds that meet the criteria, select the one with the highest specificity
    best_idx = valid_idx[np.argmax(specificity[valid_idx])]
    optimal_threshold = thresholds[best_idx]

print(f"Target minimum sensitivity: {min_sensitivity:.3f}")
print(f"Final optimal threshold: {optimal_threshold:.4f}")
print(f"Corresponding sensitivity: {tpr[best_idx]:.4f} | Specificity: {specificity[best_idx]:.4f}")

#Train the final model on the complete train+validation set
final_model = clone(rf_pipeline)
final_model.fit(X_tv, y_tv)

# Generate predictions on the independent test set using the optimal threshold
print("\n=== Final performance on the independent test set (using the optimal threshold) ===")

proba_test = final_model.predict_proba(X_test)[:, 1]
pred_test = (proba_test >= optimal_threshold).astype(int)   

auc = roc_auc_score(y_test, proba_test)
acc = accuracy_score(y_test, pred_test)
f1 = f1_score(y_test, pred_test)
prec = precision_score(y_test, pred_test)
rec = recall_score(y_test, pred_test)

tn, fp, fn, tp = confusion_matrix(y_test, pred_test).ravel()
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

print(f"AUC: {auc:.4f}")
print(f"Accuracy: {acc:.1%}")
print(f"F1-Score: {f1:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall (Sensitivity): {rec:.4f}")
print(f"Sensitivity: {sensitivity:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"|Sensitivity - Specificity|: {abs(sensitivity - specificity):.4f}")
print(f"Confusion Matrix → TP={tp}  FN={fn}  TN={tn}  FP={fp}")

In [ ]:
# SHAP analysis
import shap
import matplotlib.pyplot as plt

# Prepare background data (required by KernelExplainer)
background = shap.kmeans(X, 50)     

# Create KernelExplainer
explainer = shap.KernelExplainer(
    final_model.predict_proba,     
    background,
    link="logit"                   
)

# Calculate SHAP values ​​on the independent test set
shap_values = explainer.shap_values(X_test)  

# `shap_values` is a list; for an SVM binary classification task, `shap_values[1]` contains the SHAP values ​​for the positive class (label=1)
shap_values_pos = shap_values[1] if isinstance(shap_values, list) else shap_values

# Generate a SHAP bar plot (ranking feature importance)
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values_pos, X_test, feature_names=selected_features, plot_type="bar", show=False, max_display=10)
plt.title("SHAP Feature Importance (SVM)", fontsize=14)
plt.tight_layout()
plt.savefig("*******.pdf", dpi=300, bbox_inches='tight')
plt.show()

# Output the top 10 most important features
feature_importance = np.abs(shap_values_pos).mean(0)
top_idx = np.argsort(feature_importance)[::-1][:10]
print("\n=== Top 10 most important features (SVM) ===")
for i, idx in enumerate(top_idx):
    print(f"{i+1:2d}. {selected_features[idx]:<30}  SHAP = {feature_importance[idx]:.4f}")

In [ ]:
# read pkl file
import pandas as pd
df_final = pd.read_pickle('********.pkl')
df_final

In [ ]:
selected_features = [
    
 'max_Heart Rate',
       'max_Non Invasive Blood Pressure diastolic',
       'max_Non Invasive Blood Pressure mean',
       'max_Non Invasive Blood Pressure systolic',
       'max_O2 saturation pulseoxymetry', 'max_Respiratory Rate',
       'max_Temperature Fahrenheit', 'mean_Heart Rate',
       'mean_Non Invasive Blood Pressure diastolic',
       'mean_Non Invasive Blood Pressure mean',
       'mean_Non Invasive Blood Pressure systolic',
       'mean_O2 saturation pulseoxymetry', 'mean_Respiratory Rate',
       'mean_Temperature Fahrenheit', 'median_Heart Rate',
       'median_Non Invasive Blood Pressure diastolic',
       'median_Non Invasive Blood Pressure mean',
       'median_Non Invasive Blood Pressure systolic',
       'median_O2 saturation pulseoxymetry', 'median_Respiratory Rate',
       'median_Temperature Fahrenheit', 'min_Heart Rate',
       'min_Non Invasive Blood Pressure diastolic',
       'min_Non Invasive Blood Pressure mean',
       'min_Non Invasive Blood Pressure systolic',
       'min_O2 saturation pulseoxymetry', 'min_Respiratory Rate',
       'min_Temperature Fahrenheit', 'q25_Heart Rate',
       'q25_Non Invasive Blood Pressure diastolic',
       'q25_Non Invasive Blood Pressure mean',
       'q25_Non Invasive Blood Pressure systolic',
       'q25_O2 saturation pulseoxymetry', 'q25_Respiratory Rate',
       'q25_Temperature Fahrenheit', 'q75_Heart Rate',
       'q75_Non Invasive Blood Pressure diastolic',
       'q75_Non Invasive Blood Pressure mean',
       'q75_Non Invasive Blood Pressure systolic',
       'q75_O2 saturation pulseoxymetry', 'q75_Respiratory Rate',
       'q75_Temperature Fahrenheit', 'variance_Heart Rate',
       'variance_Non Invasive Blood Pressure diastolic',
       'variance_Non Invasive Blood Pressure mean',
       'variance_Non Invasive Blood Pressure systolic',
       'variance_O2 saturation pulseoxymetry', 'variance_Respiratory Rate',
       'variance_Temperature Fahrenheit', 
       

 ]

In [ ]:
import numpy as np
X = df_final[selected_features].values
y = df_final['label'].values
groups = df_final['subject_id'].values

print(f"Dataset shape: {X.shape}")
print(f"Class distribution: {np.bincount(y)}")
print(f"Number of unique subjects: {len(np.unique(groups))}")

In [ ]:
import numpy as np
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_curve, roc_auc_score, accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import clone
from sklearn.svm import SVC

# Data preparation
X = df_final[selected_features].values
y = df_final['label'].values
groups = df_final['subject_id'].values

# Split the data by subject to create an independent test set
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_val_idx, test_idx = next(gss.split(X, y, groups))

X_tv = X[train_val_idx]
y_tv = y[train_val_idx]
groups_tv = groups[train_val_idx]

X_test = X[test_idx]
y_test = y[test_idx]
groups_test = groups[test_idx]

print(f"Train + Validation Set: {len(np.unique(groups_tv))} subjects, {len(y_tv)} records")
print(f"Independent Test Set:   {len(np.unique(groups_test))} subjects, {len(y_test)} records")
print(f"Proportion of positive cases in the test set: {y_test.mean():.1%}")

# Build a pipeline using fixed optimal parameters
rf_base = SVC(class_weight='balanced', probability=True, random_state=42, kernel='linear', C=0.1, degree=2)

rf_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', rf_base)
])

#Perform out-of-fold (OOF) analysis on the train+validation set to determine the optimal threshold

outer_gkf = GroupKFold(n_splits=5)
oof_proba = np.zeros(len(y_tv))

for train_idx, val_idx in outer_gkf.split(X_tv, y_tv, groups_tv):
    X_train = X_tv[train_idx]
    y_train = y_tv[train_idx]
    g_train = groups_tv[train_idx]
    
    model = clone(rf_pipeline)
    model.fit(X_train, y_train)
    oof_proba[val_idx] = model.predict_proba(X_tv[val_idx])[:, 1]

# Compute the ROC curve
fpr, tpr, thresholds = roc_curve(y_tv, oof_proba)
specificity = 1 - fpr


# Specify the minimum required sensitivity
min_sensitivity = 0.62       

# Filter for all thresholds that satisfy sensitivity ≥ `min_sensitivity`
valid_idx = np.where(tpr >= min_sensitivity)[0]

if len(valid_idx) == 0:
    print("Warning: No threshold meets the minimum sensitivity requirement; please lower the `min_sensitivity` value")
    optimal_threshold = 0.5
else:
    # Among the thresholds that meet the criteria, select the one with the highest specificity
    best_idx = valid_idx[np.argmax(specificity[valid_idx])]
    optimal_threshold = thresholds[best_idx]

print(f"Target minimum sensitivity: {min_sensitivity:.3f}")
print(f"Final optimal threshold: {optimal_threshold:.4f}")
print(f"Corresponding sensitivity: {tpr[best_idx]:.4f} | Specificity: {specificity[best_idx]:.4f}")

# Train the final model on the complete train+validation set
final_model = clone(rf_pipeline)
final_model.fit(X_tv, y_tv)

# Generate predictions on the independent test set using the optimal threshold
print("\n=== Final performance on the independent test set (using the optimal threshold) ===")

proba_test = final_model.predict_proba(X_test)[:, 1]
pred_test = (proba_test >= optimal_threshold).astype(int) 


auc = roc_auc_score(y_test, proba_test)
acc = accuracy_score(y_test, pred_test)
f1 = f1_score(y_test, pred_test)
prec = precision_score(y_test, pred_test)
rec = recall_score(y_test, pred_test)

tn, fp, fn, tp = confusion_matrix(y_test, pred_test).ravel()
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

print(f"AUC: {auc:.4f}")
print(f"Accuracy: {acc:.1%}")
print(f"F1-Score: {f1:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall (Sensitivity): {rec:.4f}")
print(f"Sensitivity: {sensitivity:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"|Sensitivity - Specificity|: {abs(sensitivity - specificity):.4f}")
print(f"Confusion Matrix → TP={tp}  FN={fn}  TN={tn}  FP={fp}")

In [ ]:
# SHAP analysis
import shap
import matplotlib.pyplot as plt

# Prepare background data 
background = shap.kmeans(X, 50)     

# Create KernelExplainer
explainer = shap.KernelExplainer(
    final_model.predict_proba,      
    background,
    link="logit"                    
)

# Calculate SHAP values ​​on the independent test set
shap_values = explainer.shap_values(X_test)  

# `shap_values` is a list; for an SVM binary classification task, `shap_values[1]` contains the SHAP values ​​for the positive class (label=1)
shap_values_pos = shap_values[1] if isinstance(shap_values, list) else shap_values


#Generate a SHAP bar plot (ranking feature importance)
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values_pos, X_test, feature_names=selected_features, plot_type="bar", show=False, max_display=10)
plt.title("SHAP Feature Importance (SVM)", fontsize=14)
plt.tight_layout()
plt.savefig("********.pdf", dpi=300, bbox_inches='tight')
plt.show()

# Output the top 10 most important features
feature_importance = np.abs(shap_values_pos).mean(0)
top_idx = np.argsort(feature_importance)[::-1][:10]
print("\n=== Top 10 most important features (SVM)  ===")
for i, idx in enumerate(top_idx):
    print(f"{i+1:2d}. {selected_features[idx]:<30}  SHAP = {feature_importance[idx]:.4f}")